# 07 — Hardware Security Modules and Trusted Platform Modules

## What This Is
HSMs (Hardware Security Modules) are dedicated cryptographic processors that protect key material from extraction. TPMs (Trusted Platform Modules) are embedded chips that provide measured boot, attestation, and key storage on commodity hardware.

## Why It Matters
HSMs are the root of trust for CAs (Certificate Authorities), banks, and HSM-backed cloud KMS services. TPMs underpin Windows BitLocker, macOS FileVault, remote attestation in confidential computing, and Chrome's device trust. Without hardware roots of trust, software security is built on sand.

## Where the Community Stands
TPM 2.0 (ISO/IEC 11889:2015) is standard on all modern PCs. FIPS 140-3 Level 3/4 certifies HSMs (Thales Luna, Entrust nShield). Cloud equivalents: AWS CloudHSM, Azure Dedicated HSM, Google Cloud HSM. Confidential computing (AMD SEV-SNP, Intel TDX) extends TPM concepts to VMs.

## Key Properties
- **Key non-extractability**: private keys never leave the HSM in plaintext
- **Tamper response**: physically detected intrusion triggers key zeroing
- **Audit log**: all operations logged and non-repudiable
- **Attestation**: prove to a remote party what software is running

In [ ]:
import hashlib
import hmac
import secrets
import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# --- Simulated HSM ---

@dataclass
class HSMKey:
    key_id: str
    algorithm: str
    usage:     List[str]     # ['sign', 'verify', 'encrypt', 'decrypt']
    exportable: bool
    _key_material: bytes = field(repr=False, default=None)

class SimulatedHSM:
    """Models an HSM's key management and crypto operation interface."""

    SUPPORTED_ALGORITHMS = {'AES-256-GCM', 'RSA-2048', 'EC-P256', 'HMAC-SHA256'}

    def __init__(self, auth_token: str):
        self._auth       = auth_token
        self._keys: Dict[str, HSMKey] = {}
        self._audit_log: List[Dict] = []
        self._session_active = False

    def _log(self, operation: str, key_id: Optional[str] = None,
              result: str = 'SUCCESS') -> None:
        entry = {'op': operation, 'key': key_id, 'result': result,
                 'seq': len(self._audit_log)}
        self._audit_log.append(entry)

    def authenticate(self, token: str) -> bool:
        ok = hmac.compare_digest(token, self._auth)
        self._session_active = ok
        self._log('AUTHENTICATE', result='SUCCESS' if ok else 'FAIL')
        return ok

    def _check_session(self) -> None:
        if not self._session_active:
            raise PermissionError('No active HSM session — authenticate first')

    def generate_key(self, key_id: str, algorithm: str,
                     usage: List[str], exportable: bool = False) -> str:
        self._check_session()
        if algorithm not in self.SUPPORTED_ALGORITHMS:
            raise ValueError(f'Unsupported algorithm: {algorithm}')
        # Generate key material (simulated — real HSM uses FIPS DRBG)
        key_bytes = secrets.token_bytes(32)
        key = HSMKey(key_id=key_id, algorithm=algorithm,
                     usage=usage, exportable=exportable,
                     _key_material=key_bytes)
        self._keys[key_id] = key
        self._log('GENERATE_KEY', key_id)
        return key_id

    def hmac_sign(self, key_id: str, data: bytes) -> bytes:
        self._check_session()
        key = self._keys.get(key_id)
        if key is None:
            raise KeyError(f'Key not found: {key_id}')
        if 'sign' not in key.usage:
            raise PermissionError(f'Key {key_id} not authorised for sign')
        sig = hmac.new(key._key_material, data, hashlib.sha256).digest()
        self._log('HMAC_SIGN', key_id)
        return sig

    def export_key(self, key_id: str) -> Optional[bytes]:
        self._check_session()
        key = self._keys.get(key_id)
        if key is None:
            raise KeyError(f'Key not found: {key_id}')
        if not key.exportable:
            self._log('EXPORT_KEY', key_id, result='DENIED')
            raise PermissionError(f'Key {key_id} is non-extractable')
        self._log('EXPORT_KEY', key_id)
        return key._key_material

    def get_audit_log(self) -> List[Dict]:
        return list(self._audit_log)

# Demo
hsm = SimulatedHSM(auth_token='hsm-admin-secret-2024')
hsm.authenticate('hsm-admin-secret-2024')
hsm.generate_key('signing-key-001', 'HMAC-SHA256', ['sign', 'verify'], exportable=False)
hsm.generate_key('wrap-key-001',    'AES-256-GCM',  ['encrypt', 'decrypt'], exportable=True)

msg = b'transaction: pay $1000 to alice'
sig = hsm.hmac_sign('signing-key-001', msg)
print(f'Signed message:   {msg}')
print(f'Signature (hex):  {sig.hex()[:32]}...')

try:
    hsm.export_key('signing-key-001')
except PermissionError as e:
    print(f'Export blocked:   {e}')

print('\nAudit log:')
for entry in hsm.get_audit_log():
    print(f'  [{entry["seq"]}] {entry["op"]:<20} key={entry["key"]}  result={entry["result"]}')

## TPM 2.0: Platform Configuration Registers (PCRs)
PCRs are 256-bit registers in the TPM that record a hash chain of everything measured during boot. Each PCR starts at 0; extending it with value V produces SHA256(current_PCR || V). BitLocker seals the disk encryption key to specific PCR values — key is only released if boot measurements match expected values.

In [ ]:
class SimulatedTPM:
    """Models TPM 2.0 PCR bank and key sealing."""

    N_PCRS = 24

    def __init__(self):
        self.pcrs: List[bytes] = [b'\x00'*32] * self.N_PCRS
        self.sealed_keys: Dict[str, Dict] = {}

    def pcr_extend(self, pcr_idx: int, value: bytes) -> bytes:
        """PCR[i] = SHA256(PCR[i] || value)"""
        if not (0 <= pcr_idx < self.N_PCRS):
            raise IndexError(f'PCR index out of range: {pcr_idx}')
        current  = self.pcrs[pcr_idx]
        new_val  = hashlib.sha256(current + value).digest()
        self.pcrs[pcr_idx] = new_val
        return new_val

    def pcr_read(self, pcr_idx: int) -> bytes:
        return self.pcrs[pcr_idx]

    def seal(self, key_id: str, secret: bytes, pcr_policy: Dict[int, bytes]) -> None:
        """Seal secret against PCR values. Only unseal if PCRs match."""
        # Verify current PCRs match policy
        for idx, expected in pcr_policy.items():
            if self.pcrs[idx] != expected:
                raise ValueError(f'PCR[{idx}] mismatch — cannot seal')
        self.sealed_keys[key_id] = {
            'secret':     secret,
            'pcr_policy': pcr_policy,
        }

    def unseal(self, key_id: str) -> bytes:
        """Return secret only if current PCR values match seal policy."""
        if key_id not in self.sealed_keys:
            raise KeyError(f'Key not found: {key_id}')
        entry  = self.sealed_keys[key_id]
        policy = entry['pcr_policy']
        for idx, expected in policy.items():
            if self.pcrs[idx] != expected:
                raise PermissionError(
                    f'Unseal failed: PCR[{idx}] changed (boot tampering detected?)')
        return entry['secret']

tpm = SimulatedTPM()

# Simulate measured boot
UEFI_FW_HASH  = hashlib.sha256(b'UEFI-firmware-v2.1').digest()
BOOTLOADER    = hashlib.sha256(b'grub2-2.06').digest()
KERNEL_HASH   = hashlib.sha256(b'linux-6.5.0').digest()

tpm.pcr_extend(0, UEFI_FW_HASH)    # PCR 0: BIOS/UEFI
tpm.pcr_extend(4, BOOTLOADER)       # PCR 4: bootloader
tpm.pcr_extend(8, KERNEL_HASH)      # PCR 8: kernel

# Record policy = current PCR state after good boot
policy = {0: tpm.pcr_read(0), 4: tpm.pcr_read(4), 8: tpm.pcr_read(8)}

DISK_KEY = secrets.token_bytes(32)
tpm.seal('bitlocker-key', DISK_KEY, policy)

print(f'PCR[0] (UEFI):       {tpm.pcr_read(0).hex()[:16]}...')
print(f'PCR[4] (bootloader): {tpm.pcr_read(4).hex()[:16]}...')
print(f'PCR[8] (kernel):     {tpm.pcr_read(8).hex()[:16]}...')

# Normal boot: unseal succeeds
recovered = tpm.unseal('bitlocker-key')
print(f'\nNormal boot unseal: {"SUCCESS" if recovered == DISK_KEY else "FAIL"}')

# Tampered boot: someone loaded a different kernel
tpm.pcr_extend(8, hashlib.sha256(b'EVIL-KERNEL').digest())
try:
    tpm.unseal('bitlocker-key')
    print('Tampered boot unseal: SUCCESS (PROBLEM!)')
except PermissionError as e:
    print(f'Tampered boot unseal: BLOCKED — {e}')

## Remote Attestation
Attestation allows a remote verifier to cryptographically verify what software is running on a device. The TPM signs a quote (PCR values + nonce) with its Attestation Identity Key (AIK) — the verifier checks the signature and PCR values against known-good policy.

In [ ]:
def tpm_quote(tpm: SimulatedTPM, pcr_indices: List[int],
               nonce: bytes, aik_key: bytes) -> Dict:
    """Generate a TPM quote: signed PCR readings with nonce."""
    pcr_values = {i: tpm.pcr_read(i).hex() for i in pcr_indices}
    quote_data = json.dumps({'pcrs': pcr_values, 'nonce': nonce.hex()},
                             sort_keys=True).encode()
    signature  = hmac.new(aik_key, quote_data, hashlib.sha256).digest()
    return {'pcr_values': pcr_values, 'nonce': nonce.hex(),
            'signature': signature.hex(), 'quote_data': quote_data.hex()}

def verify_attestation(quote: Dict, expected_pcrs: Dict[int, str],
                        nonce: bytes, aik_key: bytes) -> bool:
    """Verifier checks: signature valid AND PCRs match policy."""
    # Verify signature
    quote_data = bytes.fromhex(quote['quote_data'])
    expected_sig = hmac.new(aik_key, quote_data, hashlib.sha256).digest()
    if not hmac.compare_digest(expected_sig, bytes.fromhex(quote['signature'])):
        return False
    # Verify nonce (prevents replay)
    if quote['nonce'] != nonce.hex():
        return False
    # Verify PCR values match policy
    for idx, expected_val in expected_pcrs.items():
        if quote['pcr_values'].get(str(idx)) != expected_val:
            return False
    return True

# Reset TPM and simulate a clean boot for attestation
tpm2 = SimulatedTPM()
tpm2.pcr_extend(0, UEFI_FW_HASH)
tpm2.pcr_extend(4, BOOTLOADER)
tpm2.pcr_extend(8, KERNEL_HASH)

AIK_KEY  = secrets.token_bytes(32)
nonce    = secrets.token_bytes(16)
quote    = tpm_quote(tpm2, [0, 4, 8], nonce, AIK_KEY)

good_policy = {0: tpm2.pcr_read(0).hex(), 4: tpm2.pcr_read(4).hex(), 8: tpm2.pcr_read(8).hex()}
result   = verify_attestation(quote, good_policy, nonce, AIK_KEY)
print(f'Remote attestation (clean boot): {"TRUSTED" if result else "UNTRUSTED"}')

# Replay attack (stale nonce) — must fail
stale_result = verify_attestation(quote, good_policy, secrets.token_bytes(16), AIK_KEY)
print(f'Replay attack (wrong nonce):     {"TRUSTED" if stale_result else "UNTRUSTED (replay blocked)"}')